In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.27 - SCHEMA CHANGE SCENARIO")
print("=" * 70)

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"

SOURCE_FILE = DATA_DIR / "sales_2026-08.csv"

OUTPUT_DIR = DATA_DIR / "schema_change_adjusted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "sales_2026-08_SCHEMA_CHANGE.csv"

print("\nSOURCE FILE")
print("-" * 70)
print(SOURCE_FILE)

print("\nOUTPUT FILE")
print("-" * 70)
print(OUTPUT_FILE)

# ============================================================
# 2. LOAD BASELINE AUGUST FILE
# ============================================================

df = pd.read_csv(SOURCE_FILE)

print("\nBASELINE INFORMATION")
print("-" * 70)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nOriginal columns:")
print(df.columns.tolist())

# ============================================================
# 3. VALIDATE BASELINE
# ============================================================

required_columns = [
    "sales_line_id",
    "order_id",
    "order_date",
    "order_time",
    "customer_id",
    "product_id",
    "store_id",
    "salesperson_id",
    "promotion_id",
    "sales_channel",
    "quantity",
    "unit_selling_price",
    "discount_pct",
    "payment_method",
    "order_status"
]

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required baseline columns: {missing_required}"
    )

print("\nBASELINE SCHEMA VALIDATION")
print("-" * 70)
print("Required columns present:", len(missing_required) == 0)

# ============================================================
# 4. CREATE CONTROLLED SCHEMA CHANGE
# ============================================================

schema_change_df = df.copy()

# New column required by the simulation specification
schema_change_df["coupon_code"] = pd.NA

# Identify discounted transactions
discounted_mask = (
    pd.to_numeric(
        schema_change_df["discount_pct"],
        errors="coerce"
    ).fillna(0) > 0
)

discounted_indices = schema_change_df.index[discounted_mask]

# Populate a controlled subset of discounted transactions
rng = np.random.default_rng(202608)

subset_size = int(len(discounted_indices) * 0.20)

selected_indices = rng.choice(
    discounted_indices,
    size=subset_size,
    replace=False
)

# Create coupon codes
coupon_codes = [
    f"CPN{100001 + i}"
    for i in range(subset_size)
]

schema_change_df.loc[
    selected_indices,
    "coupon_code"
] = coupon_codes

# ============================================================
# 5. SAVE CONTROLLED SCHEMA-CHANGE FILE
# ============================================================

schema_change_df.to_csv(
    OUTPUT_FILE,
    index=False
)

# ============================================================
# 6. VALIDATION
# ============================================================

saved_df = pd.read_csv(
    OUTPUT_FILE,
    keep_default_na=False
)

expected_columns = required_columns + ["coupon_code"]

column_count_valid = (
    len(saved_df.columns) == len(required_columns) + 1
)

required_columns_valid = all(
    col in saved_df.columns
    for col in required_columns
)

coupon_column_present = (
    "coupon_code" in saved_df.columns
)

row_count_preserved = (
    len(saved_df) == len(df)
)

original_ids_preserved = (
    saved_df["sales_line_id"].astype(str).tolist()
    == df["sales_line_id"].astype(str).tolist()
)

promotion_id_preserved = (
    saved_df["promotion_id"].astype(str).tolist()
    == df["promotion_id"].astype(str).tolist()
)

discounted_count = discounted_mask.sum()

coupon_populated_count = (
    saved_df["coupon_code"].astype(str).str.strip().ne("").sum()
)

coupon_subset_valid = (
    coupon_populated_count == subset_size
    and coupon_populated_count <= discounted_count
)

# Check that coupon codes exist only on discounted transactions
coupon_rows = saved_df["coupon_code"].astype(str).str.strip().ne("")

discount_values = pd.to_numeric(
    saved_df["discount_pct"],
    errors="coerce"
).fillna(0)

coupon_discount_valid = (
    (discount_values[coupon_rows] > 0).all()
    if coupon_rows.any()
    else True
)

# ============================================================
# 7. FINAL RESULT
# ============================================================

final_validation = all([
    column_count_valid,
    required_columns_valid,
    coupon_column_present,
    row_count_preserved,
    original_ids_preserved,
    promotion_id_preserved,
    coupon_subset_valid,
    coupon_discount_valid
])

print("\nSCHEMA CHANGE SUMMARY")
print("-" * 70)

print("Baseline rows:", len(df))
print("Schema-change rows:", len(saved_df))

print("Baseline columns:", len(df.columns))
print("Schema-change columns:", len(saved_df.columns))

print("New column:", "coupon_code")
print("Discounted transactions:", discounted_count)
print("Coupon codes populated:", coupon_populated_count)

print("\nVALIDATION")
print("-" * 70)

print("Row count preserved:", row_count_preserved)
print("Required columns preserved:", required_columns_valid)
print("New coupon_code column present:", coupon_column_present)
print("sales_line_id preserved:", original_ids_preserved)
print("promotion_id preserved:", promotion_id_preserved)
print("Coupons only on discounted transactions:", coupon_discount_valid)

print("\nOUTPUT")
print("-" * 70)
print("File created:", OUTPUT_FILE)
print("File exists:", OUTPUT_FILE.exists())

print("\nFINAL VALIDATION")
print("-" * 70)

if final_validation:
    print("02.27 Step 1-6 validation: PASSED")
    print("Controlled schema-change file created successfully.")
else:
    print("02.27 validation: NEEDS REVIEW")


02.27 - SCHEMA CHANGE SCENARIO

SOURCE FILE
----------------------------------------------------------------------
C:\Projects\retail_sales_analytics\data\synthetic\sales_2026-08.csv

OUTPUT FILE
----------------------------------------------------------------------
C:\Projects\retail_sales_analytics\data\synthetic\schema_change_adjusted\sales_2026-08_SCHEMA_CHANGE.csv



BASELINE INFORMATION
----------------------------------------------------------------------
Rows: 70200
Columns: 15

Original columns:
['sales_line_id', 'order_id', 'order_date', 'order_time', 'customer_id', 'product_id', 'store_id', 'salesperson_id', 'promotion_id', 'sales_channel', 'quantity', 'unit_selling_price', 'discount_pct', 'payment_method', 'order_status']

BASELINE SCHEMA VALIDATION
----------------------------------------------------------------------
Required columns present: True



SCHEMA CHANGE SUMMARY
----------------------------------------------------------------------
Baseline rows: 70200
Schema-change rows: 70200
Baseline columns: 15
Schema-change columns: 16
New column: coupon_code
Discounted transactions: 13165
Coupon codes populated: 2633

VALIDATION
----------------------------------------------------------------------
Row count preserved: True
Required columns preserved: True
New coupon_code column present: True
sales_line_id preserved: True
promotion_id preserved: False
Coupons only on discounted transactions: True

OUTPUT
----------------------------------------------------------------------
File created: C:\Projects\retail_sales_analytics\data\synthetic\schema_change_adjusted\sales_2026-08_SCHEMA_CHANGE.csv
File exists: True

FINAL VALIDATION
----------------------------------------------------------------------
02.27 validation: NEEDS REVIEW
